# SupeRR Selector Season 2 — Refreshed Phases 1 to 3

This notebook rebuilds the audit, reconstruction, and structural-pressure model from the hackathon's updated `ball_by_ball updated.csv`. It supersedes Phase 1/2 interpretations that treated `extras1_runs` as an encoded category.

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

from phase3_structural_pressure import run_pipeline
from validate_phase3_outputs import validate as validate_phase3

ROOT = Path.cwd()
DATA_DIR = ROOT / 'SupeRR Selector Season 2 | Data Pack'
PHASE1_DIR = ROOT / 'phase1_outputs'
PHASE2_DIR = ROOT / 'phase2_outputs'
PHASE3_DIR = ROOT / 'phase3_outputs'
UPDATED_BALL_FILE = 'ball_by_ball updated.csv'
SOURCE_FILES = {
    'ball_by_ball': UPDATED_BALL_FILE,
    'playing_eleven': 'playing_eleven.csv',
    'matches': 'matches.csv',
    'competition_team_player_mapping': 'competition_team_player_mapping.csv',
    'teams': 'teams.csv',
    'competitions': 'competitions.csv',
}

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def write_csv(frame, path):
    frame.to_csv(path, index=False, float_format='%.10g')

def manifest_for(directory, names, version):
    rows = []
    for name in names:
        path = directory / name
        frame = pd.read_csv(path, low_memory=False)
        rows.append({
            'artifact': name, 'rows': len(frame), 'columns': len(frame.columns),
            'bytes': path.stat().st_size, 'sha256': sha256(path), 'version': version,
        })
    return pd.DataFrame(rows)

missing = [name for name in SOURCE_FILES.values() if not (DATA_DIR / name).exists()]
require(not missing, f'Missing source files: {missing}')
data = {name: pd.read_csv(DATA_DIR / filename, low_memory=False) for name, filename in SOURCE_FILES.items()}
ball = data['ball_by_ball']
playing = data['playing_eleven']
matches = data['matches']
mapping = data['competition_team_player_mapping']
teams = data['teams']
competitions = data['competitions']
print(f'Loaded updated source: {len(ball):,} deliveries, SHA-256 {sha256(DATA_DIR / UPDATED_BALL_FILE)}')

## Phase 1 — Data understanding and audit

In [ ]:
PHASE1_DIR.mkdir(parents=True, exist_ok=True)
inventory = pd.DataFrame([
    {
        'dataset': name, 'source_file': SOURCE_FILES[name], 'rows': len(frame),
        'columns': len(frame.columns), 'exact_duplicate_rows': int(frame.duplicated().sum()),
        'sha256': sha256(DATA_DIR / SOURCE_FILES[name]),
    }
    for name, frame in data.items()
])
dictionary_rows = []
for name, frame in data.items():
    for column in frame.columns:
        dictionary_rows.append({
            'dataset': name, 'column': column, 'dtype': str(frame[column].dtype),
            'rows': len(frame), 'non_null': int(frame[column].notna().sum()),
            'null_count': int(frame[column].isna().sum()),
            'unique_non_null': int(frame[column].nunique(dropna=True)),
        })
data_dictionary = pd.DataFrame(dictionary_rows)

target_competitions = competitions.loc[
    competitions['match_format'].eq('T20I')
    | (competitions['match_format'].eq('T20') & competitions['comp_name'].str.startswith('IPL '))
].copy()
target_matches = matches.loc[matches['comp_id'].isin(target_competitions['comp_id'])].merge(
    target_competitions[['comp_id', 'comp_name', 'match_format', 'season']],
    on='comp_id', how='left', validate='many_to_one',
)
eligible_matches = target_matches.loc[~target_matches['result'].isin(['No Result', 'Match Abandoned'])].copy()
eligible_ids = set(eligible_matches['match_id'])
eligible_all = ball.loc[ball['match_id'].isin(eligible_ids)].copy()
regulation = eligible_all.loc[eligible_all['innings'].isin([1, 2])].copy()
super_overs = eligible_all.loc[eligible_all['innings'].isin([3, 4])].copy()

old_ball = pd.read_csv(DATA_DIR / 'ball_by_ball.csv', low_memory=False)
require(old_ball.shape == ball.shape, 'Updated and original delivery shapes differ unexpectedly')
key = ['match_id', 'innings', 'innings_ball_num']
require(old_ball[key].equals(ball[key]), 'Updated delivery keys/order differ from original')
change_rows = []
for column in ball.columns:
    equal = old_ball[column].eq(ball[column]) | (old_ball[column].isna() & ball[column].isna())
    if (~equal).any():
        change_rows.append({'column': column, 'changed_cells': int((~equal).sum())})
source_change = pd.DataFrame(change_rows)

identity = pd.concat([
    ball[['striker_id', 'striker_name']].rename(columns={'striker_id': 'player_id', 'striker_name': 'player_name'}),
    ball[['non_striker_id', 'non_striker_name']].rename(columns={'non_striker_id': 'player_id', 'non_striker_name': 'player_name'}),
    ball[['bowler_id', 'bowler_name']].rename(columns={'bowler_id': 'player_id', 'bowler_name': 'player_name'}),
    mapping[['player_id', 'player_name']],
], ignore_index=True).dropna()
id_name_conflicts = identity.groupby('player_id')['player_name'].nunique().gt(1)
name_id_conflicts = identity.groupby('player_name')['player_id'].nunique().gt(1)

quality_checks = pd.DataFrame([
    ('six_sources_loaded', len(data) == 6, len(data), '6 source datasets'),
    ('delivery_schema_preserved', ball.shape == old_ball.shape, str(ball.shape), str(old_ball.shape)),
    ('delivery_key_unique', not ball.duplicated(key).any(), int(ball.duplicated(key).sum()), '0 duplicates'),
    ('updated_key_order_preserved', old_ball[key].equals(ball[key]), True, True),
    ('target_matches', target_matches['match_id'].nunique() == 853, target_matches['match_id'].nunique(), 853),
    ('eligible_matches', eligible_matches['match_id'].nunique() == 813, eligible_matches['match_id'].nunique(), 813),
    ('regulation_records', len(regulation) == 192235, len(regulation), 192235),
    ('super_over_records', len(super_overs) == 26, len(super_overs), 26),
    ('all_delivery_matches_known', set(ball['match_id']).issubset(set(matches['match_id'])), len(set(ball['match_id']) - set(matches['match_id'])), 0),
    ('extras1_nonnegative', ball['extras1_runs'].ge(0).all(), int(ball['extras1_runs'].min()), '>= 0'),
    ('updated_extras_semantics', ball.loc[ball['extras_type'].isna(), 'extras1_runs'].eq(0).mean() > 0.999, float(ball.loc[ball['extras_type'].isna(), 'extras1_runs'].eq(0).mean()), '> 0.999'),
], columns=['check_name', 'passed', 'observed', 'acceptance_rule'])
quality_checks['status'] = np.where(quality_checks['passed'], 'PASS', 'FAIL')
require(quality_checks['passed'].all(), 'Phase 1 quality gate failed')

phase1_files = {
    'phase1_source_inventory.csv': inventory,
    'phase1_data_dictionary.csv': data_dictionary,
    'phase1_data_quality_summary.csv': quality_checks,
    'phase1_source_change_audit.csv': source_change,
}
for name, frame in phase1_files.items():
    write_csv(frame, PHASE1_DIR / name)
phase1_manifest = manifest_for(PHASE1_DIR, list(phase1_files), 'phase1_refresh_v2')
write_csv(phase1_manifest, PHASE1_DIR / 'phase1_output_manifest.csv')

phase1_report = f'''# SupeRR Selector Season 2 — Phase 1 Data Audit

## Status and scope

Phase 1 was rerun from all six supplied datasets. The official delivery input is `ball_by_ball updated.csv` (SHA-256 `{sha256(DATA_DIR / UPDATED_BALL_FILE)}`). The audit covers schemas, relationships, identity, match eligibility, delivery fields, missingness, and source changes. Pressure modelling and player scoring remain outside Phase 1.

## Dataset inventory

| Dataset | Rows | Columns | Exact duplicate rows |
| --- | ---: | ---: | ---: |
| Updated ball by ball | {len(ball):,} | {len(ball.columns)} | {int(ball.duplicated().sum()):,} |
| Playing eleven | {len(playing):,} | {len(playing.columns)} | {int(playing.duplicated().sum()):,} |
| Matches | {len(matches):,} | {len(matches.columns)} | {int(matches.duplicated().sum()):,} |
| Competition-team-player mapping | {len(mapping):,} | {len(mapping.columns)} | {int(mapping.duplicated().sum()):,} |
| Teams | {len(teams):,} | {len(teams.columns)} | {int(teams.duplicated().sum()):,} |
| Competitions | {len(competitions):,} | {len(competitions.columns)} | {int(competitions.duplicated().sum()):,} |
| **Total** | **{sum(len(frame) for frame in data.values()):,}** | **{sum(len(frame.columns) for frame in data.values())} across files** | **0 in each file** |

The updated delivery file contains {len(ball):,} rows and {len(ball.columns)} columns. Its match/innings/record key is unique, and every delivery match ID exists in the match registry. The machine-readable data dictionary profiles every field in every source.

## Updated-source comparison

The updated and original delivery files have identical shapes, columns, delivery keys, and row order. The update changes 199,980 `extras1_runs` cells, 24 striker-name cells, and 33 non-striker-name cells. `extras1_runs` must now be interpreted as an extras-run quantity, not as an extras-category code. `extras_type` remains the event-category field.

## Match coverage and eligibility

The match registry contains {matches.match_id.nunique():,} matches; the delivery feed contains {ball.match_id.nunique():,}; the playing-eleven source contains {playing.match_id.nunique():,}. All delivery and playing-eleven match IDs reconcile to the registry. Matches without deliveries are abandoned matches.

| Population | Matches | Treatment |
| --- | ---: | --- |
| Target IPL and Men's T20I | {target_matches.match_id.nunique():,} | Scope population |
| Eligible completed matches | {eligible_matches.match_id.nunique():,} | Include |
| IPL eligible | {int((eligible_matches.match_format == 'T20').sum()):,} | Include |
| T20I eligible | {int((eligible_matches.match_format == 'T20I').sum()):,} | Include |
| No Result or abandoned | {target_matches.match_id.nunique() - eligible_matches.match_id.nunique():,} | Exclude from completed-match analysis |

The eligible population contains {len(regulation):,} regulation delivery records from innings 1 and 2. The {len(super_overs):,} innings 3/4 records are retained separately as Super Overs and are not mixed into regulation analysis.

## Player identity and mappings

There are {identity.player_id.nunique():,} observed player IDs and {identity.player_name.nunique():,} names across delivery roles and the registration mapping. Player ID remains the identity key. One cross-source alias remains for player 4458: the updated delivery feed uses `Aneurin Donald`, while the mapping still uses `AHT Donald`. Seven names map to multiple IDs and remain namesakes rather than automatic merges. No player IDs are merged on name alone.

Playing-eleven registrations remain a squad/appearance source rather than a literal clean list of exactly 11 unique rows per side. Repeated registrations, non-playing rows, impact-player fields, and missing played batting orders remain source-level evidence and are not silently rewritten.

## Runs, extras, boundaries, and wickets

The corrected `extras1_runs` is consistent with a run quantity: no-extra rows almost always contain zero, wides range from 1–5, oversteps contain 1, and byes/leg-byes carry recorded run quantities. One row has no `extras_type` but non-zero extras fields and remains an explicit source exception. `extras_2_runs` is preserved because its precise relationship to the primary extras field is not fully documented.

Boundary flags and run values are preserved independently. Dismissal types and dismissed-player IDs are audited in Phase 2; missing or contradictory source values are not imputed.

## Legal-ball limitation

Exact legal-ball reconstruction remains unsupported for 257 `other`/penalty records. Phase 2 therefore continues to preserve evidence categories instead of inventing legal-ball status.

## Validation and deliverables

All {len(quality_checks)} Phase 1 gates pass. The refreshed data dictionary, source inventory, source-change audit, quality table, and SHA-256 manifest are in `phase1_outputs/`. These artifacts establish the approved handoff to Phase 2.
'''
(ROOT / 'Phase 1 Report.md').write_text(phase1_report, encoding='utf-8')
print(quality_checks[['check_name', 'status']].to_string(index=False))

## Phase 2 — Cleaning and reconstruction

In [ ]:
PHASE2_DIR.mkdir(parents=True, exist_ok=True)
dismissal_classified = eligible_all.copy()
dismissal_classified['dismissal_type_original'] = dismissal_classified['dismissal_type']
dismissal_classified['dismissal_type_normalized'] = dismissal_classified['dismissal_type'].replace({'Obstracting the field': 'Obstructing the field'})

legal = regulation.copy()
event_map = {
    'wide': 'RECORDED_WIDE', 'Overstep': 'RECORDED_OVERSTEP',
    'other': 'RECORDED_OTHER', 'Penalty': 'RECORDED_PENALTY',
    'byes': 'RECORDED_BYES', 'legbyes': 'RECORDED_LEGBYES',
}
legal['legal_ball_evidence_class'] = legal['extras_type'].map(event_map).fillna('RECORDED_NO_EXTRAS')
legal['legal_ball_evidence_status'] = np.select(
    [legal['extras_type'].isin(['wide', 'Overstep']), legal['extras_type'].isin(['other', 'Penalty'])],
    ['EXPLICIT_NON_LEGAL_EVIDENCE', 'UNRESOLVED'],
    default='NO_RECORDED_WIDE_OR_OVERSTEP',
)
legal['legal_ball_final_treatment'] = np.select(
    [legal['extras_type'].isin(['wide', 'Overstep']), legal['extras_type'].isin(['other', 'Penalty'])],
    ['NON_LEGAL_EVIDENCE', 'UNRESOLVED'],
    default='NO_RECORDED_WIDE_OR_OVERSTEP',
)

over_sequence = legal.groupby(['match_id', 'innings', 'overid'], observed=True).agg(
    delivery_count=('over_ball_num', 'size'),
    unique_over_ball_num=('over_ball_num', 'nunique'),
    min_over_ball_num=('over_ball_num', 'min'),
    max_over_ball_num=('over_ball_num', 'max'),
    min_innings_ball_num=('innings_ball_num', 'min'),
    max_innings_ball_num=('innings_ball_num', 'max'),
).reset_index()
over_sequence['over_ball_num_contiguous'] = (
    over_sequence['unique_over_ball_num'].eq(over_sequence['delivery_count'])
    & over_sequence['min_over_ball_num'].eq(1)
    & over_sequence['max_over_ball_num'].eq(over_sequence['delivery_count'])
)
over_profile = legal.groupby(['match_id', 'innings', 'overid'], observed=True).agg(
    delivery_count=('match_id', 'size'),
    recorded_wides=('extras_type', lambda x: int(x.eq('wide').sum())),
    recorded_oversteps=('extras_type', lambda x: int(x.eq('Overstep').sum())),
).reset_index()
unexplained_keys = over_profile.loc[
    over_profile['delivery_count'].gt(6)
    & over_profile['recorded_wides'].eq(0)
    & over_profile['recorded_oversteps'].eq(0),
    ['match_id', 'innings', 'overid'],
]
unexplained_deliveries = legal.merge(unexplained_keys, on=['match_id', 'innings', 'overid'], how='inner')
unexplained_register = unexplained_deliveries.groupby(['match_id', 'innings', 'overid'], observed=True).agg(
    delivery_count=('match_id', 'size'),
    event_classes=('extras_type', lambda x: int(x.fillna('NO_EXTRAS').nunique())),
    recorded_other=('extras_type', lambda x: int(x.eq('other').sum())),
    recorded_byes=('extras_type', lambda x: int(x.eq('byes').sum())),
    recorded_legbyes=('extras_type', lambda x: int(x.eq('legbyes').sum())),
    recorded_no_extras=('extras_type', lambda x: int(x.isna().sum())),
    extras_2_nonzero_deliveries=('extras_2_runs', lambda x: int(x.ne(0).sum())),
    extras_2_total=('extras_2_runs', 'sum'),
    runs_total=('runs', 'sum'),
).reset_index()

all_with_comp = eligible_all.merge(matches[['match_id', 'comp_id']], on='match_id', how='left', validate='many_to_one')
map_keys = set(map(tuple, mapping[['comp_id', 'team_id', 'player_id']].drop_duplicates().to_numpy()))
identity_rows, identity_exceptions = [], []
for role, team_col in [('striker_id', 'batting_team_id'), ('non_striker_id', 'batting_team_id'), ('bowler_id', 'bowling_team_id'), ('dismissed_player_id', 'batting_team_id')]:
    part = all_with_comp.loc[all_with_comp[role].notna(), ['comp_id', team_col, role]].copy()
    tuples = list(map(tuple, part.to_numpy()))
    matched = np.array([item in map_keys for item in tuples], dtype=bool)
    identity_rows.append({
        'delivery_role': role, 'expected_team': team_col, 'records_checked': len(part),
        'matched_records': int(matched.sum()), 'unmatched_records': int((~matched).sum()),
        'distinct_unmatched_players': int(part.loc[~matched, role].nunique()),
    })
    if (~matched).any():
        bad = part.loc[~matched].rename(columns={team_col: 'team_id', role: 'player_id'})
        bad['delivery_role'] = role
        identity_exceptions.append(bad)
team_identity_audit = pd.DataFrame(identity_rows)
team_identity_exceptions = (
    pd.concat(identity_exceptions, ignore_index=True)
    if identity_exceptions else pd.DataFrame(columns=['comp_id', 'team_id', 'player_id', 'delivery_role'])
)
require(team_identity_audit['unmatched_records'].eq(0).all(), 'Team/player identity validation failed')

master = dismissal_classified.loc[dismissal_classified['innings'].isin([1, 2])].merge(
    legal[key + ['legal_ball_evidence_class', 'legal_ball_evidence_status', 'legal_ball_final_treatment']],
    on=key, how='left', validate='one_to_one',
)
match_context = [
    'match_id', 'comp_id', 'match_number', 'match_date', 'stage', 'venue_id', 'venue_country',
    'home_team_id', 'home_team', 'away_team_id', 'away_team', 'toss_win_team_id', 'toss_opted',
    'win_team_id', 'result', 'revised_target', 'first_inn_revised_overs',
    'second_inn_revised_overs', 'knock_out',
]
master = master.merge(eligible_matches[match_context], on='match_id', how='left', validate='many_to_one')
master = master.merge(competitions[['comp_id', 'comp_name', 'match_format', 'season']], on='comp_id', how='left', validate='many_to_one')
master = master.merge(
    teams[['comp_id', 'team_id', 'team_name']].rename(columns={'team_id': 'batting_team_id', 'team_name': 'batting_team_name'}),
    on=['comp_id', 'batting_team_id'], how='left', validate='many_to_one',
)
master = master.merge(
    teams[['comp_id', 'team_id', 'team_name']].rename(columns={'team_id': 'bowling_team_id', 'team_name': 'bowling_team_name'}),
    on=['comp_id', 'bowling_team_id'], how='left', validate='many_to_one',
)

exception_rows = []
def add_exception(exception_id, exception_type, dataset, affected_records, field, observation, treatment, match_id=np.nan, innings=np.nan, overid=np.nan, innings_ball_num=np.nan):
    exception_rows.append({
        'exception_id': exception_id, 'exception_type': exception_type, 'dataset': dataset,
        'match_id': match_id, 'innings': innings, 'overid': overid, 'innings_ball_num': innings_ball_num,
        'affected_records': affected_records, 'field': field, 'observation': observation,
        'treatment': treatment, 'status': 'Documented - Unresolved',
    })
missing_order = playing.loc[playing['is_played'].eq(True) & playing['batting_order'].isna()]
add_exception('P2-XI-001', 'MISSING_PLAYED_BATTING_ORDER', 'playing_eleven', len(missing_order), 'batting_order', 'Played registrations have missing batting order.', 'Preserve; do not impute.', match_id=missing_order['match_id'].iloc[0])
for index, row in legal.loc[legal['extras_type'].eq('other') & legal['extras_2_runs'].ne(0)].reset_index(drop=True).iterrows():
    add_exception(f'P2-EXTRAS2-{index + 1:03d}', 'NONZERO_EXTRAS2_WITH_RECORDED_OTHER', 'ball_by_ball', 1, 'extras_2_runs', 'Non-zero extras_2_runs with extras_type other.', 'Preserve source value; semantics unresolved.', row.match_id, row.innings, row.overid, row.innings_ball_num)
inconsistent_extra = legal.loc[legal['extras_type'].isna() & (legal['extras1_runs'].ne(0) | legal['extras_2_runs'].ne(0))]
for index, row in inconsistent_extra.reset_index(drop=True).iterrows():
    add_exception(f'P2-EXTRATYPE-{index + 1:03d}', 'MISSING_EXTRAS_TYPE_WITH_RUNS', 'ball_by_ball', 1, 'extras_type', 'Extras runs are populated while extras_type is missing.', 'Preserve and exclude from assumptions requiring a known extras category.', row.match_id, row.innings, row.overid, row.innings_ball_num)
missing_dismissed = dismissal_classified.loc[dismissal_classified['is_wicket'] & dismissal_classified['dismissed_player_id'].isna()]
for index, row in missing_dismissed.reset_index(drop=True).iterrows():
    add_exception(f'P2-WICKET-{index + 1:03d}', 'WICKET_MISSING_DISMISSED_PLAYER', 'ball_by_ball', 1, 'dismissed_player_id', 'Wicket record has no dismissed player ID.', 'Preserve; do not impute.', row.match_id, row.innings, row.overid, row.innings_ball_num)
nonwicket_dismissed = dismissal_classified.loc[~dismissal_classified['is_wicket'] & dismissal_classified['dismissed_player_id'].notna()]
for index, row in nonwicket_dismissed.reset_index(drop=True).iterrows():
    add_exception(f'P2-NONWICKET-{index + 1:03d}', 'NONWICKET_WITH_DISMISSED_PLAYER', 'ball_by_ball', 1, 'dismissed_player_id', 'Non-wicket record has a dismissed player ID.', 'Preserve; do not reclassify.', row.match_id, row.innings, row.overid, row.innings_ball_num)
variant_count = int(dismissal_classified['dismissal_type'].eq('Obstracting the field').sum())
add_exception('P2-DISMISSAL-001', 'DISMISSAL_LABEL_VARIANT', 'ball_by_ball', variant_count, 'dismissal_type', 'Misspelled source label Obstracting the field.', 'Normalize analytical copy; preserve original field.')
add_exception('P2-LEGAL-001', 'LEGAL_BALL_RECONSTRUCTION_LIMITATION', 'ball_by_ball', int(legal['legal_ball_final_treatment'].eq('UNRESOLVED').sum()), 'extras_type', 'Other and penalty records do not establish definitive legal-ball status.', 'Retain unresolved evidence class; do not infer legal balls.')
exceptions = pd.DataFrame(exception_rows)

phase2_checks = pd.DataFrame([
    ('regulation_rows', len(master) == 192235, len(master), 192235),
    ('eligible_matches', master['match_id'].nunique() == 813, master['match_id'].nunique(), 813),
    ('master_key_unique', not master.duplicated(key).any(), int(master.duplicated(key).sum()), 0),
    ('master_source_keys_exact', set(map(tuple, master[key].to_numpy())) == set(map(tuple, regulation[key].to_numpy())), len(master), len(regulation)),
    ('legal_treatment_complete', master['legal_ball_final_treatment'].notna().all(), int(master['legal_ball_final_treatment'].isna().sum()), 0),
    ('explicit_nonlegal_records', master['legal_ball_final_treatment'].eq('NON_LEGAL_EVIDENCE').sum() == 7976, int(master['legal_ball_final_treatment'].eq('NON_LEGAL_EVIDENCE').sum()), 7976),
    ('unresolved_legal_records', master['legal_ball_final_treatment'].eq('UNRESOLVED').sum() == 257, int(master['legal_ball_final_treatment'].eq('UNRESOLVED').sum()), 257),
    ('over_sequences_contiguous', over_sequence['over_ball_num_contiguous'].all(), int((~over_sequence['over_ball_num_contiguous']).sum()), 0),
    ('unexplained_extended_overs', len(unexplained_register) == 147, len(unexplained_register), 147),
    ('team_identity_complete', team_identity_audit['unmatched_records'].eq(0).all(), int(team_identity_audit['unmatched_records'].sum()), 0),
    ('context_complete', master[['comp_name', 'match_format', 'batting_team_name', 'bowling_team_name']].notna().all().all(), int(master[['comp_name', 'match_format', 'batting_team_name', 'bowling_team_name']].isna().sum().sum()), 0),
], columns=['check_name', 'passed', 'observed', 'acceptance_rule'])
phase2_checks['status'] = np.where(phase2_checks['passed'], 'PASS', 'FAIL')
require(phase2_checks['passed'].all(), 'Phase 2 validation failed')

phase2_files = {
    'phase2_integrated_cleaned_deliveries.csv': master,
    'phase2_exception_register.csv': exceptions,
    'phase2_legal_ball_evidence.csv': legal,
    'phase2_unexplained_extended_over_register.csv': unexplained_register,
    'phase2_dismissal_classified.csv': dismissal_classified,
    'phase2_team_identity_audit.csv': team_identity_audit,
    'phase2_team_identity_exceptions.csv': team_identity_exceptions,
    'phase2_over_sequence_audit.csv': over_sequence,
    'phase2_validation_gates.csv': phase2_checks,
}
for name, frame in phase2_files.items():
    write_csv(frame, PHASE2_DIR / name)
try:
    master.to_parquet(PHASE2_DIR / 'phase2_integrated_cleaned_deliveries.parquet', index=False)
except (ImportError, ModuleNotFoundError):
    pass
phase2_manifest = manifest_for(PHASE2_DIR, list(phase2_files), 'phase2_refresh_v2')
phase2_manifest['source_file'] = UPDATED_BALL_FILE
phase2_manifest['source_sha256'] = sha256(DATA_DIR / UPDATED_BALL_FILE)
write_csv(phase2_manifest, PHASE2_DIR / 'phase2_export_manifest.csv')
completion = pd.DataFrame([
    ('Target matches', target_matches['match_id'].nunique()),
    ('Eligible matches', master['match_id'].nunique()),
    ('Eligible regulation deliveries', len(master)),
    ('Super Over deliveries kept separate', len(super_overs)),
    ('Integrated dataset columns', len(master.columns)),
    ('Exception register entries', len(exceptions)),
    ('Unexplained extended overs', len(unexplained_register)),
    ('Unresolved legal-ball deliveries', int(master['legal_ball_final_treatment'].eq('UNRESOLVED').sum())),
    ('Explicit wide/overstep records', int(master['legal_ball_final_treatment'].eq('NON_LEGAL_EVIDENCE').sum())),
], columns=['metric', 'value'])
write_csv(completion, PHASE2_DIR / 'phase2_completion_summary.csv')

phase2_report = f'''# SupeRR Selector Season 2 — Phase 2 Data Reconstruction

## Executive summary

Phase 2 was rebuilt from `ball_by_ball updated.csv`. The validated master contains **{len(master):,}** regulation delivery records, **{master.match_id.nunique():,}** eligible matches, **{master[['match_id', 'innings']].drop_duplicates().shape[0]:,}** innings, and **{len(master.columns)}** columns. All {len(phase2_checks)} validation gates pass.

The rebuild preserves source values, normalizes only the documented dismissal spelling variant, derives legal-ball evidence from `extras_type`, attaches verified match/competition/team context, and records unresolved cases instead of imputing them.

## Population and scope

| Measure | Value |
| --- | ---: |
| Target matches | {target_matches.match_id.nunique():,} |
| Eligible matches | {master.match_id.nunique():,} |
| Excluded incomplete/no-result matches | {target_matches.match_id.nunique() - master.match_id.nunique():,} |
| Regulation delivery records | {len(master):,} |
| Super Over records kept separately | {len(super_overs):,} |
| Competitions represented | {master.comp_id.nunique():,} |
| Integrated columns | {len(master.columns)} |

Super Overs are retained in the dismissal and identity evidence outputs but excluded from the regulation master used by Phase 3.

## Corrected runs and extras treatment

`extras1_runs` is now treated as a numeric extras-run quantity. `extras_type` supplies the event category. The prior code mapping (1=overstep, 2=wide, and so on) is superseded and is not used anywhere in the refreshed pipeline.

Wide and `Overstep` records provide explicit non-legal evidence (**{int(master.legal_ball_final_treatment.eq('NON_LEGAL_EVIDENCE').sum()):,}** records). Byes, leg-byes, and rows without an extras category are not marked non-legal. The **{int(master.legal_ball_final_treatment.eq('UNRESOLVED').sum()):,}** `other`/penalty records remain unresolved rather than being guessed.

| Evidence treatment | Records |
| --- | ---: |
| Explicit non-legal evidence: wide/overstep | {int(master.legal_ball_final_treatment.eq('NON_LEGAL_EVIDENCE').sum()):,} |
| No recorded wide or overstep | {int(master.legal_ball_final_treatment.eq('NO_RECORDED_WIDE_OR_OVERSTEP').sum()):,} |
| Unresolved other/penalty | {int(master.legal_ball_final_treatment.eq('UNRESOLVED').sum()):,} |
| **Total** | **{len(master):,}** |

One regulation row has no `extras_type` despite non-zero extras fields. It is preserved and separately registered. Non-zero `extras_2_runs` values are also preserved because their exact semantics are not fully documented.

## Legal-ball and over-sequence evidence

There are **{len(unexplained_register):,}** overs with more than six delivery records but no recorded wide or overstep. This is why Phase 2 still does not claim definitive legal-ball progression.

All **{len(over_sequence):,}** recorded over sequences have unique, contiguous `over_ball_num` values from 1 through their recorded delivery count. `innings_ball_num` and `over_ball_num` remain delivery-record sequences, not guaranteed legal-ball counters.

## Dismissals and identities

All striker, non-striker, bowler, and populated dismissed-player competition/team/player mappings reconcile. The analytical copy normalizes `Obstracting the field` to `Obstructing the field` while retaining `dismissal_type_original`. One wicket lacks a dismissed-player ID, and {len(nonwicket_dismissed):,} non-wicket records have populated dismissed-player IDs; neither condition is silently changed.

Player IDs remain the analytical identity key. Updated delivery names flow through the master, including `Aneurin Donald`; the mapping-table alias `AHT Donald` remains documented in Phase 1 rather than forcing an ID or name rewrite.

## Integrated master and consistency

The master retains all 33 updated delivery columns, adds original/normalized dismissal labels and three legal-ball evidence fields, and attaches match, competition, season, and team names. It has zero duplicate delivery keys, zero missing competition/team context values, and no unexpected or missing approved matches.

## Exception register

The refreshed exception register contains **{len(exceptions):,}** audit entries. It covers {len(missing_order):,} played registrations with missing batting order, {int(legal['extras_type'].eq('other').mul(legal['extras_2_runs'].ne(0)).sum()):,} individual `other` records with secondary extras, {len(inconsistent_extra):,} missing-extras-type inconsistency, {len(missing_dismissed):,} wicket with a missing dismissed-player ID, {len(nonwicket_dismissed):,} non-wicket dismissed-player records, the dismissal spelling variant, and the aggregate legal-ball limitation. Entries may overlap in their underlying delivery evidence.

## Validation

The validation gates confirm exact source-key preservation, approved population counts, complete evidence classifications, expected wide/overstep and unresolved counts, contiguous over sequences, complete player/team mappings, and complete attached context. The CSV outputs were reread during downstream validation, and the manifest records every file's dimensions, bytes, and SHA-256 hash.

## Deliverables

The refreshed master, dismissal table, legal-ball evidence, over audit, unexplained-over register, identity audit, exception register, validation gates, completion summary, and hash manifest are in `phase2_outputs/`. The manifest records the official updated source name and SHA-256 hash.

## Downstream limitations

Phase 3 must not infer exact legal balls or balls remaining from unresolved records. Revised/shortened matches require separate treatment. Phase 4 may use batter runs and verified dismissal outcomes under its locked record filters, but must not relabel its run-suppression metric as conventional economy.
'''
(ROOT / 'Phase 2 report.md').write_text(phase2_report, encoding='utf-8')
print(phase2_checks[['check_name', 'status']].to_string(index=False))

## Phase 3 — Structural pressure refresh

In [ ]:
phase3_results = run_pipeline(
    PHASE2_DIR / 'phase2_integrated_cleaned_deliveries.csv',
    PHASE3_DIR,
    ROOT / 'Phase 3 Structural Pressure Report.md',
)
independent_phase3 = validate_phase3(PHASE2_DIR / 'phase2_integrated_cleaned_deliveries.csv', PHASE3_DIR)
require(independent_phase3['status'].eq('PASS').all(), 'Independent Phase 3 validation failed')

consolidated_report = f'''# SupeRR Selector Season 2 — Consolidated Report (Phases 0–3)

## Executive decision and refresh status

Phases 1–3 were rebuilt from the official updated delivery source. Source: `ball_by_ball updated.csv`; SHA-256 `{sha256(DATA_DIR / UPDATED_BALL_FILE)}`. Phase 0 methodology governance remains unchanged. The updated file corrects extras-run values but does not eliminate the unresolved legal-ball cases, so the validated structural-pressure Phase 3 remains the official model.

The authoritative downstream files are `phase2_outputs/phase2_integrated_cleaned_deliveries.csv` and `phase3_outputs/phase3_structural_pressure_labelled.csv`. Root-level score/chase artifacts from the superseded implementation remain non-authoritative.

## Phase 0 — Methodology governance

The following rules remain binding: pressure is a pre-delivery state; IPL and T20I retain separate baselines; batter and bowler share the same pressure state on a delivery; unsupported cricket facts are not imputed; eligibility and reliability precede ranking; and uncertainty belongs in Phase 5.

## Phase 1 — Data audit

The six sources contain {sum(len(frame) for frame in data.values()):,} total records. The approved population remains {eligible_matches.match_id.nunique():,} matches and {len(regulation):,} regulation delivery records. The updated file preserves schema and keys but corrects `extras1_runs` into a run quantity and changes delivery-name observations from AHT Donald to Aneurin Donald.

| Population | Matches/records |
| --- | ---: |
| Target IPL and Men's T20I matches | {target_matches.match_id.nunique():,} |
| Eligible completed matches | {eligible_matches.match_id.nunique():,} |
| Regulation delivery records | {len(regulation):,} |
| Super Over records kept separately | {len(super_overs):,} |

Player ID remains the identity key. The mapping-table alias for player 4458 is documented rather than merged or overwritten. All {len(quality_checks)} Phase 1 gates pass, and the data dictionary and source-change audit are persisted in `phase1_outputs/`.

## Phase 2 — Validated analytical master

The refreshed analytical master contains {len(master):,} rows. Legal-ball evidence is now derived from `extras_type`, never from numeric `extras1_runs` values. There are 7,976 explicit wide/overstep records and 257 unresolved other/penalty records. All {len(phase2_checks)} Phase 2 validation gates pass.

| Evidence treatment | Records |
| --- | ---: |
| Explicit non-legal: wide/overstep | {int(master.legal_ball_final_treatment.eq('NON_LEGAL_EVIDENCE').sum()):,} |
| No recorded wide or overstep | {int(master.legal_ball_final_treatment.eq('NO_RECORDED_WIDE_OR_OVERSTEP').sum()):,} |
| Unresolved other/penalty | {int(master.legal_ball_final_treatment.eq('UNRESOLVED').sum()):,} |

The {len(unexplained_register):,} extended overs without a recorded wide/overstep prevent a definitive legal-ball reconstruction. All delivery keys, over sequences, identity mappings, and attached context values pass their checks. The master preserves updated source values and records all exceptions explicitly.

## Phase 3 — Structural pressure model

The structural-pressure model remains appropriate because exact legal-ball progression is still unsupported. It labels {len(phase3_results['deliveries']):,} delivery records across {phase3_results['deliveries'].match_id.nunique():,} unrevised matches and excludes {len(phase3_results['excluded_matches'])} revised/shortened matches. All {len(phase3_results['validation'])} pipeline gates and {len(independent_phase3)} independent validation checks pass.

For each delivery, the model uses only state known before the current outcome: wickets already lost, wickets in the prior 12 delivery records, and source-over stage urgency. Expected wickets are leave-one-match-out by format, innings, and source over, with minimum support 100 and declared fallback levels.

The components are bounded as follows: positive wicket-strain z-scores are capped at 3 SD and divided by 3; recent wicket shock is prior-window wickets divided by 3 and capped at 1; stage urgency is `(overid - 1) / 19`. Their equal-weight mean is the continuous structural-pressure index. Low/Medium/High categories are descriptive within-format-and-innings percentile bands; the continuous index remains primary.

The primary dataset contains {len(phase3_results['benchmarks']):,} match-state benchmark rows. Benchmark support ranges from {int(phase3_results['benchmarks'].benchmark_support.min()):,} to {int(phase3_results['benchmarks'].benchmark_support.max()):,}. Sensitivity variants retain a minimum non-primary Spearman correlation of {phase3_results['sensitivity'].loc[phase3_results['sensitivity'].variant.ne('primary_equal_w12'), 'spearman_vs_primary'].min():.4f}.

## Validation and Phase 4 handoff

The refresh is reproducible through the consolidated notebook. Phase 3 manifest hashes are independently recomputed. Phase 4 must use only the authoritative pressure-labelled file, retain the pre-delivery construction, exclude revised matches, evaluate IPL and T20I separately, and preserve the full-period/no-recency-weight decision.
'''
(ROOT / 'Phase_0_to_3_Consolidated_Report.md').write_text(consolidated_report, encoding='utf-8')
print(phase3_results['validation'][['check_name', 'status']].to_string(index=False))
print(independent_phase3.to_string(index=False))
print('PHASES_1_TO_3_REFRESH_OK')